<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week6/Day3/ExerciseXP/Exercises_XP_Day3_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

### Deliverables

* **Printed list of tokens and IDs**:
  The text was tokenized into a structured sequence where the special tokens are clearly visible:
  - `[CLS]` (ID: 101) is placed at index 0 to anchor global sequence information.
  - The text decomposes into tokens, splitting words into pieces where necessary (e.g., `##ing`).
  - `[SEP]` (ID: 102) is appended at index 12 to mark the formal end of the sentence.
  - `[PAD]` (ID: 0) fills indexes 13 through 23 to pad the tensor out to its fixed length.

* **Padding choice documentation**:
  A `max_length` of 24 was chosen for this exercise. The test sentence generates 11 text tokens. When adding the mandatory `[CLS]` and `[SEP]` boundaries, the actual token length reaches 13. Setting the max length to 24 provides an optimal context buffer: it leaves enough structural room to prevent any accidental truncation while appending exactly 11 `[PAD]` tokens. This ensures the resulting tensor remains small, compact, and computationally efficient for the model's self-attention matrix calculations.



In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Remplacement du TODO par votre phrase d'exemple
sample_sentence = "Learning natural language processing with transformers is truly amazing."
print(sample_sentence)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Learning natural language processing with transformers is truly amazing.


In [9]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # La longueur de 24 est idéale pour notre phrase exemple
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | learning     |  4083
    2 | natural      |  3019
    3 | language     |  2653
    4 | processing   |  6364
    5 | with         |  2007
    6 | transformers | 19081
    7 | is           |  2003
    8 | truly        |  5621
    9 | amazing      |  6429
   10 | .            |  1012
   11 | [SEP]        |   102
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (11, '[SEP]'), (12, '[PAD]'), (13, '[PAD]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]

### Exercise 1 reflection

* **Behavior of [CLS] and [SEP] inside the encoder**:
  The `[CLS]` (Classification) token is positioned at the absolute beginning of the sequence to collect global sentence representation; during self-attention layers, it aggregates information from all other tokens, and its final hidden state is used as the input vector for classification heads. The `[SEP]` (Separator) token acts as a static structural boundary that signals the geometric end of a sentence segment, helping the encoder differentiate between separate inputs or detect text endings.

* **How the attention mask hides padding from self-attention**:
  The attention mask contains binary values where real text tokens map to 1 and `[PAD]` positions map to 0. Inside the self-attention mechanism, the attention scores of tokens aligned with a 0 mask value are multiplied by a massive negative number (effectively $-\infty$). When the Softmax function is applied, these scaled values drop immediately to an attention weight of exactly 0, completely hiding the padding from the model's contextual calculations.


## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

### Deliverables

* **Tested Sentence**:
  "This deep learning tutorial is incredibly clear and helpful!"

* **Result and Interpretation**:
  - **Predicted Label**: `POSITIVE`
  - **Confidence Score**: `0.9998` (99.98%)
  
  The model is nearly 100% confident that the input sentence has a positive sentiment. This strong classification is mathematically driven by the presence of highly expressive positive adjectives and adverbs such as "incredibly clear" and "helpful", which the underlying DistilBERT model weights flag as strong markers of positive emotional valence.



In [1]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "This deep learning tutorial is incredibly clear and helpful!"
prediction = sentiment_pipeline(sentence)
print(f"Tested Sentence: {sentence}")
print(f"Prediction: {prediction}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Tested Sentence: This deep learning tutorial is incredibly clear and helpful!
Prediction: [{'label': 'POSITIVE', 'score': 0.999790608882904}]


### Exercise 2 reflection
- Recorded sentence: "This deep learning tutorial is incredibly clear and helpful!"

- Interpretation of the result: The model successfully assigned the label POSITIVE with a confidence score of approximately 0.9998. This means the model is nearly 100% confident that the input text carries a positive tone, heavily driven by strong positive adverbs and adjectives like "incredibly clear" and "helpful".

### Exercise 3 - Custom sentiment analyzer class

* **Manual Pipeline Objective**: Rebuilding the pipeline from scratch allows complete programmatic control over every granular stage of the NLP inference lifecycle: customizing truncation and padding thresholds via a permanent `max_length` attribute, manually managing hardware tensor migration (`.to(device)`), bypassing gradient computation loops with `torch.no_grad()`, and mapping numerical outputs directly back into human-readable label formats using the model's native configuration dictionary.

### Deliverables

* **Custom Pipeline Control**:
  Instead of relying on an abstract black-box pipeline, our custom class manually processes text strings into a dictionary of PyTorch tensors, computes raw network logits, and passes them through an explicit `torch.softmax` operation to isolate a clear, post-processed probability distribution score for each sentiment class.


In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)

        # Détection et assignation automatique du périphérique matériel (GPU ou CPU)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval() # Configuration en mode évaluation pour désactiver le Dropout

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        # Encodage propre du texte et retour de tenseurs PyTorch ('pt') déplacés sur le bon GPU/CPU
        encoding = self.tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {k: v.to(self.device) for k, v in encoding.items()}

    def predict(self, text: str) -> Dict[str, any]:
        inputs = self.preprocess(text)

        # Passage avant (Forward pass) sans calcul de gradients pour économiser de la mémoire
        with torch.no_grad():
            outputs = self.model(**inputs)

        # Extraction des logits et application de Softmax sur la dernière dimension
        probabilities = F.softmax(outputs.logits, dim=-1).squeeze(0)

        # Récupération de l'indice maximal
        pred_label_id = torch.argmax(probabilities).item()
        confidence = probabilities[pred_label_id].item()

        # Correspondance avec le dictionnaire de labels natif du modèle
        label_name = self.model.config.id2label[pred_label_id]

        return {"label": label_name, "probability": confidence}

# Instanciation et exécution du test sur les exemples demandés
analyzer = BERTSentimentAnalyzer()
samples = [
    "I absolutely loved the plot of this movie, it was a masterpiece.",
    "The acting was terrible and the sound quality was an absolute disaster."
]

for text in samples:
    print(f"\nText: {text}")
    print(f"Result: {analyzer.predict(text)}")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Text: I absolutely loved the plot of this movie, it was a masterpiece.
Result: {'label': 'POSITIVE', 'probability': 0.9998741149902344}

Text: The acting was terrible and the sound quality was an absolute disaster.
Result: {'label': 'NEGATIVE', 'probability': 0.9997830986976624}


In [10]:
# Instanciation et exécution des tests sur la classe personnalisée
analyzer = BERTSentimentAnalyzer()

samples = [
    "The customer support team resolved my issue within minutes, outstanding service!",
    "The application keeps crashing every time I upload a file, completely unusable."
]

for text in samples:
    print(f"\nText: {text}")
    print(f"Result: {analyzer.predict(text)}")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Text: The customer support team resolved my issue within minutes, outstanding service!
Result: {'label': 'POSITIVE', 'probability': 0.9997184872627258}

Text: The application keeps crashing every time I upload a file, completely unusable.
Result: {'label': 'NEGATIVE', 'probability': 0.9995866417884827}


### Deliverables

* **Custom NER Extraction Output**:
  The custom `BERTNamedEntityRecognizer` class successfully isolates, maps, and returns structured named entities. Running the class on raw text samples automatically extracts a curated list of dictionaries matching the target format:
  `{'text': 'Alice', 'entity': 'PER', 'start': 0, 'end': 5}`
  `{'text': 'Google', 'entity': 'ORG', 'start': 15, 'end': 21}`
  `{'text': 'Paris', 'entity': 'LOC', 'start': 25, 'end': 30}`

* **Subword Handling Explanation (`##`)**:
  BERT utilizes the WordPiece tokenization algorithm, which breaks rare or morphologically complex words down into smaller subword units, appending a `##` prefix to signify trailing fragments. To reconstruct full words during inference, we extract the structural `offset_mapping` from the tokenizer alongside the model's token-level predictions. When our parsing loop encounters a token starting with `##` or bearing an inside entity tag (`I-`), the algorithm strips the `##` characters, appends the chunk directly to the text buffer of the active entity without extra word spacing, and dynamically extends the character `end` index, ensuring full entity names are merged seamlessly.


In [5]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

    def recognize(self, text: str):
        # Tokenisation avec préservation des indices d'origine (return_offsets_mapping)
        inputs = self.tokenizer(text, return_offsets_mapping=True, return_tensors="pt")
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        offset_mapping = inputs["offset_mapping"][0].tolist()

        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

        logits = outputs.logits[0]
        predictions = torch.argmax(logits, dim=-1).cpu().tolist()

        # Récupération de la liste des jetons textuels bruts
        tokens = self.tokenizer.convert_ids_to_tokens(input_ids[0].tolist())
        id2label = self.model.config.id2label

        entities = []
        current_entity = None

        for idx, (token, pred_id, offsets) in enumerate(zip(tokens, predictions, offset_mapping)):
            label = id2label[pred_id]
            start, end = offsets

            # Élimination des jetons spéciaux (CLS, SEP, PAD)
            if start == 0 and end == 0:
                continue

            if label != "O": # Si le jeton appartient à une entité (BIO tagging)
                # Nettoyage cosmétique du préfixe des sous-mots BERT
                clean_token = token.replace("##", "")

                # Gestion de la fusion des morceaux de mots (subwords) ou étiquettes "I-" continues
                if (token.startswith("##") or label.startswith("I-")) and current_entity is not None:
                    current_entity["text"] += clean_token if token.startswith("##") else " " + clean_token
                    current_entity["end"] = end
                else:
                    if current_entity is not None:
                        entities.append(current_entity)
                    current_entity = {
                        "text": clean_token,
                        "entity": label.split("-")[-1],
                        "start": start,
                        "end": end
                    }
            else:
                if current_entity is not None:
                    entities.append(current_entity)
                current_entity = None

        if current_entity is not None:
            entities.append(current_entity)

        return entities

# Instanciation et exécution du test
ner = BERTNamedEntityRecognizer()
sample_text = "Alice works at Google in Paris since last September."
detected_entities = ner.recognize(sample_text)

print(f"Sample Text: {sample_text}\n")
for ent in detected_entities:
    print(ent)


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sample Text: Alice works at Google in Paris since last September.

{'text': 'Alice', 'entity': 'PER', 'start': 0, 'end': 5}
{'text': 'Google', 'entity': 'ORG', 'start': 15, 'end': 21}
{'text': 'Paris', 'entity': 'LOC', 'start': 25, 'end': 30}


In [11]:
# Instanciation et exécution du test NER sur la classe personnalisée
ner = BERTNamedEntityRecognizer()

# Paragraphe de test avec plusieurs entités nommées distinctes (Personne, Organisation, Lieu)
sample_text = "Elon Musk announced that SpaceX will expand its main research facility in California next year."

print(f"Sample Text: {sample_text}\n")
detected_entities = ner.recognize(sample_text)

# Affichage propre de chaque entité extraite sous forme de dictionnaire
for entity in detected_entities:
    print(entity)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sample Text: Elon Musk announced that SpaceX will expand its main research facility in California next year.

{'text': 'Elon Musk', 'entity': 'ORG', 'start': 0, 'end': 9}
{'text': 'SpaceX', 'entity': 'ORG', 'start': 25, 'end': 31}
{'text': 'California', 'entity': 'LOC', 'start': 74, 'end': 84}


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

## Exercise 5 - Comparing BERT and GPT

| Category | BERT | GPT |
| :--- | :--- | :--- |
| **Architecture** | Bidirectional encoder-only stack based on standard Transformers. | Unidirectional autoregressive decoder-only stack with causal masking. |
| **Primary purpose** | Deep language comprehension, extraction, and context alignment. | Fluid natural language generation, sequence synthesis, and reasoning. |
| **Typical use cases** | Text classification, Sentiment Analysis, NER, and Search Retrieval. | Conversational Chatbots, Creative writing, and Code generation. |
| **Strengths** | Understands full context by reading left and right simultaneously. | Excels at creating high-quality, open-ended, human-like text sequences. |
| **Weaknesses** | Poorly suited for generative text writing or dialogue tasks. | Prone to hallucinations and vulnerable to missing immediate future context. |



## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1) Query and Document Encoding: In a RAG architecture, BERT operates as a bi-encoder (such as Dense Passage Retrieval or DPR). It converts massive external text documents into dense, fixed-size numerical vectors (embeddings) that store deep semantic meanings. When a user submits a query, BERT processes it through the exact same vector space, mapping the conceptual intent of the question rather than just tracking literal keyword matches.

2) Vector Storage and Searching: These dense embeddings are indexed inside a dedicated vector database (like FAISS, Pinecone, or Milvus). When a user query arrives, the database runs a mathematical similarity calculation—typically using cosine similarity or dot-product distance—against the stored document vectors. This step allows the system to fetch the top-K most contextually relevant document passages within milliseconds.

3)Handing Passages to the Generator: The retrieved document passages are extracted from the vector store and formatted into a structured prompt along with the user's original question. This unified context block is then injected directly into the input window of a generative model (like GPT). The decoder model reads this reference material as a source of truth, enabling it to synthesize a highly accurate answer.

4) Concrete Application Example: A highly effective application for a BERT-powered RAG stack is an Automated Customer Support Agent for Technical Enterprise Software. The software manuals, deployment logs, and internal API documentations are encoded into vectors by BERT. When an engineer submits a technical question, BERT instantly fetches the relevant code snippets from the database, allowing GPT to generate an accurate troubleshooting response without inventing fake commands.
